# Sovereign AI — Mistral 7B BFSI Fine-Tuning (Colab)

QLoRA fine-tuning of **Mistral 7B** on a BFSI (Banking, Financial Services, Insurance) instruction dataset,
merging the adapter, converting to GGUF, and preparing for Ollama deployment.

**Before you start:**
- `Runtime > Change runtime type > GPU` — pick **T4** (free tier) or **A100** (Colab Pro/Pro+).
- You need a Hugging Face account + access token (read scope) to pull `mistralai/Mistral-7B-v0.1`.
- Replace the sample BFSI data in Phase 2 with your real corpus before doing a production run.

| Tier | GPU | VRAM | Notes |
|---|---|---|---|
| Free | T4 | 16GB | Use 4-bit QLoRA, `max_seq_length=512`, `batch_size=1` |
| Colab Pro | A100/V100 | 40GB | QLoRA or 8-bit, larger batch sizes |
| Colab Pro+ | A100 | 40GB | Background execution for long runs |


## Phase 1 — Environment Setup

Install libraries, authenticate with Hugging Face, mount Google Drive so data/checkpoints survive a disconnect.

In [ ]:
!pip install -q -U transformers accelerate peft bitsandbytes trl datasets sentencepiece huggingface_hub

In [ ]:
from huggingface_hub import login
login()  # paste an HF token with read access when prompted

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
WORKDIR = "/content/drive/MyDrive/sovereign-ai-bfsi"
os.makedirs(WORKDIR, exist_ok=True)
print("Working directory:", WORKDIR)

## Phase 2 — Data Collection

At Colab scale, aim for 500–5,000 instruction examples to start (not billions of tokens — that's pretraining/DAPT scale).

The cell below writes a **small sample BFSI dataset** to Drive so the notebook runs end-to-end out of the box.
**Replace this with your real corpus** (upload a JSONL file to `WORKDIR` and point `DATA_FILE` at it) before a real training run.

In [ ]:
import json

sample_data = [
    {"instruction": "Explain what NPA means for a bank's balance sheet.",
     "response": "NPA (Non-Performing Asset) is a loan or advance where the borrower has stopped making "
                  "principal or interest payments for a specified period (typically 90 days). Banks must "
                  "classify these separately and set aside provisions, which reduces reported profitability "
                  "and signals credit-quality stress on the balance sheet."},
    {"instruction": "Summarize the core purpose of KYC in banking.",
     "response": "KYC (Know Your Customer) is the process banks use to verify a customer's identity, assess "
                  "risk profile, and monitor transactions, in order to prevent fraud, money laundering, and "
                  "terrorist financing, as mandated by regulators such as RBI and FATF guidelines."},
    {"instruction": "What is the difference between CRR and SLR?",
     "response": "CRR (Cash Reserve Ratio) is the percentage of a bank's deposits that must be held as cash "
                  "reserves with the central bank, earning no interest. SLR (Statutory Liquidity Ratio) is the "
                  "percentage that must be maintained in liquid assets such as cash, gold, or approved "
                  "government securities. Both are monetary policy tools to control liquidity and credit growth."},
    {"instruction": "Define AML in the context of financial services.",
     "response": "AML (Anti-Money Laundering) refers to the laws, regulations, and procedures financial "
                  "institutions follow to detect and prevent the disguising of illegally obtained funds as "
                  "legitimate income, typically involving transaction monitoring, suspicious activity reporting, "
                  "and customer due diligence."},
    {"instruction": "What does NBFC stand for and how does it differ from a bank?",
     "response": "NBFC (Non-Banking Financial Company) provides banking-like services such as loans, asset "
                  "financing, and investment products, but cannot accept demand deposits or issue cheques "
                  "drawn on itself, and is not part of the payment/settlement system the way a licensed bank is."},
]

DATA_FILE = f"{WORKDIR}/bfsi_train.jsonl"
with open(DATA_FILE, "w") as f:
    for row in sample_data:
        f.write(json.dumps(row) + "\n")

print(f"Wrote {len(sample_data)} sample examples to {DATA_FILE}")
print("Replace this file with your real BFSI corpus before a production training run.")

In [ ]:
from datasets import load_dataset

ds = load_dataset("json", data_files=DATA_FILE)["train"]
print(ds)
ds[0]

## Phase 3 — Data Cleaning & Formatting

Format every example into Mistral's chat template. For real BFSI data, also run dedup, PII masking, and length
filtering before this step.

In [ ]:
def to_mistral_format(ex):
    return {"text": f"<s>[INST] {ex['instruction']} [/INST] {ex['response']}</s>"}

ds = ds.map(to_mistral_format)
ds = ds.filter(lambda ex: len(ex["text"]) > 20)   # drop empties
ds = ds.train_test_split(test_size=0.2, seed=42)  # small sample -> larger test fraction; use 0.1 on a real corpus
print(ds)
print(ds["train"][0]["text"])

## Phase 4 — Tokenizer Extension (32K BFSI Vocabulary)

Mistral's tokenizer already has a 32,000-token vocabulary — extend it with BFSI terms that fragment badly,
rather than building a separate tokenizer from scratch.

In [ ]:
from transformers import AutoTokenizer

MODEL_ID = "mistralai/Mistral-7B-v0.1"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
print("Base vocab size:", tokenizer.vocab_size)

bfsi_terms = ["NPA", "CASA", "IRDAI", "AML", "KYC", "moratorium", "IFSC", "ISIN", "NBFC", "SLR", "CRR"]
for t in bfsi_terms:
    print(f"{t:12s} -> {tokenizer.tokenize(t)}")

In [ ]:
# Only add tokens that fragment into more than 2 pieces today
new_tokens = [t for t in bfsi_terms if len(tokenizer.tokenize(t)) > 2]
num_added = tokenizer.add_tokens(new_tokens)
tokenizer.pad_token = tokenizer.eos_token

print(f"Added {num_added} new BFSI tokens: {new_tokens}")
print("New vocab size:", len(tokenizer))

## Phase 5 — Load Model in 4-bit (QLoRA)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

if new_tokens:
    model.resize_token_embeddings(len(tokenizer))

model.gradient_checkpointing_enable()
model.enable_input_require_grads()
print("Model loaded in 4-bit.")

## Phase 6 — LoRA Configuration

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()  # expect ~0.1-0.3% of total params trainable

## Phase 7 — Fine-Tuning (SFT)

On a free T4: keep `per_device_train_batch_size=1-2` and `max_seq_length<=512` if you hit CUDA OOM.
On an A100: batch size 4-8 and `max_seq_length=1024+` are comfortable.
`save_strategy="epoch"` checkpoints to Drive so a disconnect doesn't lose progress —
resume with `trainer.train(resume_from_checkpoint=True)`.

In [ ]:
from trl import SFTTrainer, SFTConfig

sft_config = SFTConfig(
    output_dir=f"{WORKDIR}/checkpoints",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=8,
    num_train_epochs=3,
    learning_rate=2e-4,
    bf16=True,              # set fp16=True instead if on a T4 (no native bf16)
    logging_steps=5,
    save_strategy="epoch",
    max_seq_length=512,     # raise to 1024+ on A100
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=ds["train"],
    eval_dataset=ds["test"],
    args=sft_config,
    dataset_text_field="text",
)

In [ ]:
trainer.train()

In [ ]:
ADAPTER_DIR = f"{WORKDIR}/mistral-bfsi-adapter"
trainer.save_model(ADAPTER_DIR)
tokenizer.save_pretrained(ADAPTER_DIR)
print("Adapter saved to", ADAPTER_DIR)

## Phase 8 — Evaluation

Quick qualitative check. Build a proper held-out BFSI test set (20–50 Q&A pairs) before trusting results.

In [ ]:
def ask(prompt, max_new_tokens=200):
    inputs = tokenizer(f"<s>[INST] {prompt} [/INST]", return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tokenizer.decode(out[0], skip_special_tokens=True)

test_prompts = [
    "What does CRR stand for and how does it affect bank liquidity?",
    "Summarize the key purpose of KYC in one sentence.",
]
for p in test_prompts:
    print("Q:", p)
    print("A:", ask(p))
    print("-" * 60)

## Phase 9 — Merge Adapter & Save Full Model

Merges the LoRA weights into the base model for standalone deployment (required before GGUF conversion).
This reloads the base model in fp16 — if you hit VRAM pressure, **restart the runtime** first, re-run Phase 1's
setup cell and Phase 4's tokenizer cell, then run this cell directly (no need to redo the 4-bit training load).

In [ ]:
from transformers import AutoModelForCausalLM
from peft import PeftModel
import torch

base_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map="auto"
)
if new_tokens:
    base_model.resize_token_embeddings(len(tokenizer))

merged = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
merged = merged.merge_and_unload()

MERGED_DIR = f"{WORKDIR}/mistral-bfsi-merged"
merged.save_pretrained(MERGED_DIR, safe_serialization=True)
tokenizer.save_pretrained(MERGED_DIR)
print("Merged model saved to", MERGED_DIR)

Optional: push the merged model to the Hugging Face Hub for safekeeping/sharing.

In [ ]:
# merged.push_to_hub("your-username/mistral-7b-bfsi")
# tokenizer.push_to_hub("your-username/mistral-7b-bfsi")

## Phase 10 — Convert to GGUF for Ollama

Slowest step (7B fp16 → GGUF conversion + quantization). Runs entirely in Colab; download the final
`.gguf` file to your local machine afterward.

In [ ]:
!git clone --depth 1 https://github.com/ggerganov/llama.cpp
!pip install -q -r llama.cpp/requirements.txt

In [ ]:
!python llama.cpp/convert_hf_to_gguf.py "{MERGED_DIR}" \
    --outfile "{WORKDIR}/mistral-bfsi.gguf" --outtype f16

In [ ]:
!cmake -B llama.cpp/build llama.cpp
!cmake --build llama.cpp/build --target llama-quantize -j

!./llama.cpp/build/bin/llama-quantize "{WORKDIR}/mistral-bfsi.gguf" \
    "{WORKDIR}/mistral-bfsi-Q4_K_M.gguf" Q4_K_M

print("GGUF ready — download mistral-bfsi-Q4_K_M.gguf from your Drive folder:", WORKDIR)

## Phase 11 — Deploy Locally with Ollama

Run this part **on your own machine**, not in Colab, after downloading the `.gguf` file.

Create a file named `Modelfile` next to the downloaded `.gguf`:

```
FROM ./mistral-bfsi-Q4_K_M.gguf
TEMPLATE "[INST] {{ .Prompt }} [/INST]"
SYSTEM "You are a BFSI domain assistant specialized in banking, financial services, and insurance."
PARAMETER temperature 0.3
```

Then in a terminal:

```bash
ollama create mistral-bfsi -f Modelfile
ollama run mistral-bfsi "Explain the difference between NPA and NPL."
```


## Phase 12 — Monitoring & Iteration

Keep this lightweight at Colab scale:
- Log real usage prompts/outputs to a file or spreadsheet.
- Review weekly; add corrected examples back into the Phase 2 JSONL.
- Re-run Phases 5–9 periodically (each fine-tuning run is a few hours) — monthly or quarterly retraining is
  realistic even on free-tier Colab as your BFSI corpus grows.

---
**Practical tips**
- CUDA OOM on T4 → drop `max_seq_length` to 512 (or lower), `per_device_train_batch_size` to 1, raise
  `gradient_accumulation_steps`.
- Keep raw data, checkpoints, and the merged model all under the mounted Drive folder — Colab's local disk is
  wiped on disconnect.
- If Phase 9 (merge) runs out of memory right after Phase 7 (training), restart the runtime first — loading the
  base model twice (once 4-bit, once fp16) in the same session can exhaust VRAM.
